In [1]:
#| default_exp xl

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
model_path = 'xl/pelevin'

In [6]:
#| export
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./tokenizer/rugpt3xl.tokenizer", local_files_only=True)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_2048.json",
    seq_len=seq_length,
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

[2024-09-29 16:54:48,021] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
> initializing model parallel with size 1
[2024-09-29 16:54:50,842] [INFO] [config.py:733:__init__] Config mesh_device None world_size = 1


[W929 16:54:50.582103804 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/workspaces/gpt/src/xl_wrapper.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights

In [8]:
model

RuGPT3XL(
  (model): FP16_Module(
    (module): GPT3Model(
      (word_embeddings): VocabParallelEmbedding()
      (position_embeddings): Embedding(2048, 2048)
      (embedding_dropout): Dropout(p=0.1, inplace=False)
      (transformer): GPT3ParallelTransformer(
        (layers): ModuleList(
          (0-23): 24 x GPT3ParallelTransformerLayer(
            (input_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (attention): GPT3ParallelSelfAttention(
              (query_key_value): ColumnParallelLinear()
              (attention_dropout): Dropout(p=0.1, inplace=False)
              (dense): RowParallelLinear()
              (output_dropout): Dropout(p=0.1, inplace=False)
            )
            (post_attention_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (mlp): GPT3ParallelMLP(
              (dense_h_to_4h): ColumnParallelLinear()
              (dense_4h_to_h): RowParallelLinear()
        

In [9]:
sum(p.numel() for p in model.parameters())

1315737600

In [10]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.float16,
                                 checkpoint=None,
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2024-09-29 16:55:11,920] [INFO] [logging.py:96:log_dist] [Rank -1] DeepSpeed info: version=0.15.1, git-hash=unknown, git-branch=unknown
[2024-09-29 16:55:11,921] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2024-09-29 16:55:11,922] [INFO] [logging.py:96:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [11]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=1.0):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak, temperature)

In [12]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/workspaces/gpt/src/xl_wrapper.py:280: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  context_tokens_tensor = torch.cuda.LongTensor(context_tokens)


CPU times: user 5.53 s, sys: 346 ms, total: 5.88 s
Wall time: 5.52 s


['- просто хуй с горы. Скажу тебе еще раз, не говори, что ты депутат, ты ни хуя не депутат. Если кто и может защитить права трудящихся, так только эти труженики.',
 ' хуй с палкой, – прокомментировал Т. – Прости, не сдержался». (Герой, кстати, узнал в машине, куда он едет, свой поезд.) Т., однако, слишком долго не мог успокоиться.',
 ' — бездельник!» — бросил ему вслед Федя. Он не понял, из-за кого так расстроился Ноздря. Понять не мог и еще долго не понимал, почему он так злится, когда на город опускается солнечный диск.',
 ' — нет. Или то, что с виду Толстой — на самом деле не Толстой?» — «Нет, — отвечала Танюша. — Я не рисую. Я скульптор».']

In [13]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])


- Ты сука, иди отсюда, надоел со своими расспросами. Я тебя всю жизнь ждал, дура.

CPU times: user 1.53 s, sys: 14.5 ms, total: 1.54 s
Wall time: 998 ms
